# IRI-2016 Kaggle Verification (Diagnostic Only) — revision 3

> **Revision 3 (2026-09-19) — NOT yet executed on Kaggle.** The successful Kaggle run of
> 2026-09-19 (bundle `iri_verification_bundle.zip`, SHA-256 `3a0723a1…`) was produced by
> **revision 2**, SHA-256 `b8399c98f248749fca3b6e5acebec9543c262ec0cc83dde2dab0042460d564fa`,
> preserved unchanged at `evidence/iri2016_kaggle_verification_2026-09-19/`. Nothing in
> that evidence is attributable to this file. Revision 3 adds, without changing the
> smoke test or the pins: a bounded **network preflight** before any install (Step 1b),
> a closed-set **failure classification** on every failed command, and full
> **`ig_rz.dat` / `apf107.dat` metadata, coverage and 2022-support checks** in the inner
> script (Step 4). It still writes the diagnostic bundle on every stop.


**Purpose.** Verify that the exact published `iricore` release the project selected (D-45, `governance/CHANGE_RECORD_2026-09-19_scientific_decisions.md` and `_p2.md`) actually installs and runs on Kaggle's Linux/CPU environment, with a single permitted smoke-test call at a real, approved station coordinate and a non-December synthetic timestamp -- and nothing more.

**This notebook does NOT:**
- read, download, or reference any GNSS/VTEC target data, ICTP/Madrigal files, or anything under `evidence/locked_test_restricted/`;
- run a full-year or multi-station benchmark;
- register a producer artifact, call `write_release`, or write to `permitted_producers`;
- pass G-04, certify any feature leakage-free, or constitute a scientific result.

**Everything this notebook produces is a diagnostic verification bundle** — an installation/runtime report, not a producer release or a registered benchmark artifact.

**Boundaries respected:** CPU only; no GPU requested or required; no Conda (Kaggle's own Python + a `venv` only); no credentials of any kind are embedded — this notebook reads only public PyPI package files by content hash; no access to any Windows filesystem (everything reads/writes under `/kaggle/working`).

**What is selected, and why (decided before running, not adjusted afterwards):**
- `iricore==1.8.0` — the MOST RECENT published release that ships a prebuilt **Linux** (`manylinux_2_35_x86_64`) wheel, for **CPython 3.10** (`cp310-cp310-manylinux_2_35_x86_64`). Releases 1.8.1–1.9.0 (checked exhaustively against the PyPI JSON API) publish a macOS-arm64 wheel only — no Linux wheel exists at any newer version, so this is a deliberate, verified choice, not a default or an assumption that master equals a release.
- This is **not** the `master` branch; it is the immutable PyPI release archive, pinned by its own published SHA-256 (checked below, before install).
- If Kaggle's own Python is not exactly 3.10, this notebook creates an **isolated environment** at Python 3.10 (revision 2 established on Kaggle that `virtualenv` against the image's own `/usr/bin/python3.10` works; `uv`-managed CPython 3.10 and `apt-get` remain as fallbacks) specifically so the pinned `numpy==1.26.4` (matching the project's own governed pin) and the other exact versions below are never imposed on Kaggle's own base environment. **If Python 3.10 cannot be obtained, this notebook stops with a precise diagnosis — it never silently falls back to a mismatched wheel or an unpinned install.**

In [ ]:
import datetime as dt
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
import zipfile
from pathlib import Path

BUNDLE_DIR = Path('/kaggle/working/iri_verification_bundle')
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
VENV_DIR = Path('/kaggle/working/iri_venv')

report = {
    'notebook': 'kaggle_iri2016_verification.ipynb',
    'notebook_revision': 3,
    'note': 'revision 3 has NOT produced the 2026-09-19 evidence bundle; that was revision 2 (sha256 b8399c98...)',
    'purpose': 'diagnostic installation/runtime verification only -- NOT a producer release or registered benchmark artifact',
    'generated_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'boundaries': {
        'gnss_vtec_target_data_accessed': False,
        'full_year_benchmark_run': False,
        'producer_artifact_registered': False,
        'g04_passed': False,
    },
}


def run(cmd, *, timeout=900, check=False, env=None):
    """Capture a command's real stdout/stderr/exit code -- never inferred. A timeout is
    itself a recorded outcome (returncode None, timed_out True, partial output kept),
    never an uncaught exception that would lose the report."""
    print('$', ' '.join(cmd) if isinstance(cmd, list) else cmd)
    merged_env = dict(os.environ, **(env or {}))
    entry = {'cmd': cmd if isinstance(cmd, str) else ' '.join(cmd), 'timed_out': False}
    try:
        proc = subprocess.run(
            cmd, shell=isinstance(cmd, str), capture_output=True, text=True,
            timeout=timeout, env=merged_env,
        )
        out, err, rc = proc.stdout or '', proc.stderr or '', proc.returncode
    except subprocess.TimeoutExpired as exc:
        def _s(b):
            return b.decode('utf-8', 'replace') if isinstance(b, bytes) else (b or '')
        out, err, rc = _s(exc.stdout), _s(exc.stderr), None
        entry['timed_out'] = True
        entry['timeout_seconds'] = timeout
        print(f'--- TIMED OUT after {timeout}s (recorded, not raised) ---')
    entry.update({'returncode': rc, 'stdout_tail': out[-4000:], 'stderr_tail': err[-4000:]})
    if rc != 0 or entry['timed_out']:
        # closed-set class (dns_failure / tls_failure / hash_mismatch / ...) so a reader
        # can tell failure kinds apart without the raw log; defined in Step 1b's cell
        entry['failure_class'] = classify_command_failure(err, out, rc, entry['timed_out'])
    print(out[-2000:])
    if err:
        print('--- stderr (tail) ---')
        print(err[-2000:])
    if check and rc != 0:
        raise RuntimeError(f'command failed (exit {rc}): {entry["cmd"]}')
    return entry


def save_and_stop(reason):
    """Write whatever the report holds so far, zip the bundle, then raise -- a clear
    stop with a precise diagnosis, never a silent fallback to something unverified."""
    report['ok'] = False
    report['stopped_reason'] = reason
    write_bundle()
    raise RuntimeError(f'STOPPED: {reason}')


def write_bundle():
    report_path = BUNDLE_DIR / 'verification_report.json'
    report_path.write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
    zip_path = Path('/kaggle/working/iri_verification_bundle.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in BUNDLE_DIR.rglob('*'):
            if p.is_file():
                zf.write(p, p.relative_to(BUNDLE_DIR))
    print('Bundle written:', zip_path)
    return zip_path

## Step 1 — Inspect Kaggle's ACTUAL Python/platform before installing anything

Nothing below assumes this matches the project's governed local Python 3.11.16.

In [ ]:
runtime = {
    'python_version': sys.version,
    'python_version_info': list(sys.version_info),
    'python_executable': sys.executable,
    'platform_platform': platform.platform(),
    'platform_machine': platform.machine(),
    'platform_system': platform.system(),
    'in_kaggle': Path('/kaggle').is_dir(),
}
report['runtime'] = runtime
print(json.dumps(runtime, indent=2))

## Step 1b — Bounded network preflight (before anything is installed)

Every install path below needs `pypi.org` (index) and `files.pythonhosted.org` (wheels).
This cell probes both, stage by stage — **DNS → TCP → TLS → HTTPS** — each stage under a
10 s timeout, and stops the notebook at the first failing stage with the stage named,
the exception text, and the **possible** causes. Runs 2 and 3 of revision 2 spent three
failed rungs and a 240 s `apt-get` timeout discovering what this cell reports in seconds.
The probe cannot see Kaggle's settings panel, so it never asserts that the Internet
setting is OFF; it lists that as one possible cause of a DNS failure, to check first.
The same cell defines `classify_command_failure`, used by `run()` on every failed command.


In [ ]:
"""iri_net_preflight.py -- bounded network preflight and failure classification for the
Kaggle IRI-2016 verification notebook.

Purpose: before anything is installed, establish whether the package index and the
wheel host are reachable, stage by stage (DNS -> TCP -> TLS -> HTTPS), each stage under
its own timeout, and name the first stage that fails. Also classifies the stderr of a
failed install command into a small closed set so a report reader can tell a DNS
failure from a TLS failure from a hash mismatch without reading raw logs.

Inputs: host names and paths; a captured command's stdout/stderr/exit code.
Re-run behaviour: pure network probes and pure string classification; nothing is
written; the caller records the returned dicts.
"""
import socket
import ssl
import time
import urllib.error
import urllib.request

NETWORK_FAILURE_CLASSES = (
    "dns_failure", "tcp_timeout", "tcp_connect_failure", "tls_failure", "tls_timeout",
    "http_error", "http_timeout", "unknown",
)

# Possible causes are stated as possibilities. The preflight cannot see the Kaggle
# settings panel, so it never asserts that the Internet toggle is off.
POSSIBLE_CAUSES = {
    "dns_failure": "name resolution failed: possible causes include the Kaggle notebook's "
                   "Internet setting being OFF (Settings sidebar -> Internet), a DNS outage, "
                   "or a restricted network; this probe cannot tell these apart -- check the "
                   "Internet setting first, then re-run",
    "tcp_timeout": "the name resolved but no TCP connection completed within the timeout: "
                   "possible causes include a firewall, a proxy requirement, or an outage",
    "tcp_connect_failure": "the name resolved but the TCP connection was refused or reset: "
                           "possible causes include a firewall, a proxy requirement, or an outage",
    "tls_failure": "TCP connected but the TLS handshake or certificate validation failed: "
                   "possible causes include a TLS-intercepting proxy, a stale CA bundle, or a "
                   "clock error on the machine",
    "tls_timeout": "TCP connected but the TLS handshake did not complete within the timeout",
    "http_error": "TLS succeeded but the HTTPS request returned an error status: possible "
                  "causes include a blocked path, a proxy error page, or a service incident",
    "http_timeout": "TLS succeeded but the HTTPS response did not arrive within the timeout",
    "unknown": "an unclassified network error; see the recorded exception text",
}


def _elapsed(t0):
    return round(time.monotonic() - t0, 3)


def probe_host(host, port=443, path="/", timeout=10.0, expect_status=None):
    """DNS -> TCP -> TLS -> HTTPS GET, stopping at the first failing stage.

    Returns {"host", "port", "path", "ok", "stages": {...}, "failure_class"?, "error"?}.
    Every stage records its wall time; a failing stage records repr(exception).
    """
    out = {"host": host, "port": port, "path": path, "stages": {}, "ok": False}

    t0 = time.monotonic()
    try:
        infos = socket.getaddrinfo(host, port, type=socket.SOCK_STREAM)
        addrs = sorted({i[4][0] for i in infos})
        out["stages"]["dns"] = {"ok": True, "addresses": addrs[:8], "seconds": _elapsed(t0)}
    except socket.gaierror as exc:
        out["stages"]["dns"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "dns_failure"
        out["error"] = repr(exc)
        return out

    t0 = time.monotonic()
    try:
        sock = socket.create_connection((host, port), timeout=timeout)
        out["stages"]["tcp"] = {"ok": True, "peer": list(sock.getpeername()[:2]), "seconds": _elapsed(t0)}
    except socket.timeout as exc:
        out["stages"]["tcp"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "tcp_timeout"
        out["error"] = repr(exc)
        return out
    except OSError as exc:
        out["stages"]["tcp"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "tcp_connect_failure"
        out["error"] = repr(exc)
        return out

    t0 = time.monotonic()
    try:
        ctx = ssl.create_default_context()
        sock.settimeout(timeout)
        tls = ctx.wrap_socket(sock, server_hostname=host)
        cert = tls.getpeercert() or {}
        subject = dict(x[0] for x in cert.get("subject", ())) if cert.get("subject") else {}
        out["stages"]["tls"] = {
            "ok": True, "version": tls.version(), "cipher": (tls.cipher() or ("",))[0],
            "peer_common_name": subject.get("commonName"), "not_after": cert.get("notAfter"),
            "seconds": _elapsed(t0),
        }
        tls.close()
    except socket.timeout as exc:
        out["stages"]["tls"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "tls_timeout"
        out["error"] = repr(exc)
        sock.close()
        return out
    except (ssl.SSLError, ssl.CertificateError, OSError) as exc:
        out["stages"]["tls"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "tls_failure"
        out["error"] = repr(exc)
        sock.close()
        return out

    t0 = time.monotonic()
    url = f"https://{host}{path}"
    req = urllib.request.Request(url, headers={"User-Agent": "iri-verification-preflight/3"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            status = resp.status
            resp.read(4096)
        st = {"ok": True, "status": status, "seconds": _elapsed(t0)}
        if expect_status is not None and status != expect_status:
            st.update({"ok": False, "expected_status": expect_status})
            out["stages"]["https"] = st
            out["failure_class"] = "http_error"
            out["error"] = f"HTTP {status} from {url}, expected {expect_status}"
            return out
        out["stages"]["https"] = st
    except urllib.error.HTTPError as exc:
        st = {"ok": False, "status": exc.code, "error": repr(exc), "seconds": _elapsed(t0)}
        if expect_status is None:  # any HTTP answer proves the path end to end
            st["ok"] = True
            out["stages"]["https"] = st
        else:
            out["stages"]["https"] = st
            out["failure_class"] = "http_error"
            out["error"] = repr(exc)
            return out
    except urllib.error.URLError as exc:
        out["stages"]["https"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        reason = exc.reason
        if isinstance(reason, socket.gaierror):
            out["failure_class"] = "dns_failure"
        elif isinstance(reason, (ssl.SSLError, ssl.CertificateError)):
            out["failure_class"] = "tls_failure"
        elif isinstance(reason, socket.timeout) or "timed out" in str(reason):
            out["failure_class"] = "http_timeout"
        else:
            out["failure_class"] = "unknown"
        out["error"] = repr(exc)
        return out
    except socket.timeout as exc:
        out["stages"]["https"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "http_timeout"
        out["error"] = repr(exc)
        return out

    out["ok"] = True
    return out


def network_preflight(targets=None, timeout=10.0):
    """Probe each target in order; overall ok only if every target is ok.

    Default targets: the PyPI simple index page for iricore (must answer 200) and the
    wheel host (any HTTP answer accepted). Bounded: at most 4 stages x timeout per
    target.
    """
    if targets is None:
        targets = [
            {"host": "pypi.org", "path": "/simple/iricore/", "expect_status": 200},
            {"host": "files.pythonhosted.org", "path": "/", "expect_status": None},
        ]
    results = [probe_host(t["host"], path=t["path"], timeout=timeout, expect_status=t.get("expect_status"))
               for t in targets]
    failed = [r for r in results if not r["ok"]]
    summary = {"ok": not failed, "timeout_seconds_per_stage": timeout, "targets": results}
    if failed:
        first = failed[0]
        summary["failure_class"] = first.get("failure_class", "unknown")
        summary["failed_host"] = first["host"]
        summary["failed_stage"] = next((k for k, v in first["stages"].items() if not v.get("ok")), None)
        summary["error"] = first.get("error")
        summary["possible_causes"] = POSSIBLE_CAUSES.get(summary["failure_class"], POSSIBLE_CAUSES["unknown"])
    return summary


# ---- classifying a failed install command -------------------------------------------

COMMAND_FAILURE_CLASSES = (
    "none", "timeout", "dns_failure", "tls_failure", "connection_failure", "hash_mismatch",
    "platform_tag_mismatch", "resolution_failure", "missing_module", "unknown",
)

_DNS = ("Temporary failure in name resolution", "Name or service not known",
        "nodename nor servname provided", "getaddrinfo failed", "Name resolution failure")
_TLS = ("CERTIFICATE_VERIFY_FAILED", "certificate verify failed", "SSLError", "SSL: ",
        "TLSV1_ALERT", "WRONG_VERSION_NUMBER")
_HASH = ("THESE PACKAGES DO NOT MATCH THE HASHES", "Hashes are required in --require-hashes mode",
         "do not match the hashes")
_PLATFORM = ("is not a supported wheel on this platform", "not supported on this platform")
_CONN = ("Connection refused", "Connection reset", "NewConnectionError", "Max retries exceeded",
         "ProxyError", "Network is unreachable", "ReadTimeoutError", "ConnectTimeoutError")
_RESOLUTION = ("No matching distribution found", "Could not find a version that satisfies")


def classify_command_failure(stderr, stdout="", returncode=None, timed_out=False):
    """Map a failed command's output to one closed-set class. Order matters: a DNS
    failure also prints 'No matching distribution found', so network signatures are
    tested before resolution ones."""
    if timed_out:
        return "timeout"
    if returncode == 0:
        return "none"
    text = (stderr or "") + "\n" + (stdout or "")
    if any(s in text for s in _DNS):
        return "dns_failure"
    if any(s in text for s in _TLS):
        return "tls_failure"
    if any(s in text for s in _HASH):
        return "hash_mismatch"
    if any(s in text for s in _PLATFORM):
        return "platform_tag_mismatch"
    if any(s in text for s in _CONN):
        return "connection_failure"
    if any(s in text for s in _RESOLUTION):
        return "resolution_failure"
    if "No module named" in text:
        return "missing_module"
    return "unknown"


report['network_preflight'] = network_preflight(timeout=10.0)
print(json.dumps(report['network_preflight'], indent=2, default=str))
write_bundle()
if not report['network_preflight']['ok']:
    npf = report['network_preflight']
    save_and_stop(
        f"network preflight failed before any install: host {npf['failed_host']!r}, stage "
        f"{npf['failed_stage']!r}, class {npf['failure_class']!r}: {npf['error']}. "
        f"{npf['possible_causes']}. Nothing was installed; see report['network_preflight'] "
        f"for every stage's outcome and timing."
    )


## Step 2 — Decide the environment strategy

`iricore==1.8.0`'s only Linux wheel targets CPython **3.10** exactly (`cp310-cp310-manylinux_2_35_x86_64`). This step decides, from the ACTUAL Python detected above, whether Kaggle's own kernel Python already is 3.10 (then an isolated `venv` is created from it directly, purely to keep the pinned `numpy`/`fortranformat`/`pymap3d` versions from touching Kaggle's own site-packages) or whether a separate `python3.10` binary must be obtained first (via `apt-get`, which needs Kaggle's internet toggle ON). **If neither path is available, this notebook stops here with a precise diagnosis — it does not attempt a from-source Fortran/CMake build.**

In [ ]:
venv_python_source = None
if sys.version_info[:2] == (3, 10):
    venv_python_source = sys.executable
    strategy = 'kernel_python_is_3.10_isolate_via_venv'
else:
    which = shutil.which('python3.10')
    if which:
        venv_python_source = which
        strategy = 'found_existing_python3.10_isolate_via_venv'
    else:
        # No system python3.10: Step 3 will obtain a complete CPython 3.10 through uv's
        # managed interpreters (an immutable published python-build-standalone release),
        # and only as a last resort through apt-get.
        strategy = 'no_system_python3.10_use_uv_managed_3.10'

report['environment_strategy'] = {
    'kernel_python_minor': sys.version_info[:2],
    'strategy': strategy,
    'venv_python_source': venv_python_source,
}
print(json.dumps(report['environment_strategy'], indent=2, default=str))

write_bundle()  # partial report on disk before anything is installed

## Step 3 — Create the isolated environment and install the exact pinned, hash-verified dependencies

**How the isolated environment is created (revised after the first Kaggle run).** The first run found `/usr/bin/python3.10` already present on the Kaggle image and `python3.10 -m venv` failed (exit 1): Debian/Ubuntu ship that interpreter WITHOUT the `python3.10-venv` package, so the stdlib `venv` module has no `ensurepip` and cannot seed pip. This notebook therefore uses **`virtualenv`** — installed into the kernel's own Python via its working pip — which bundles its own pip/setuptools wheels and needs nothing from the target interpreter beyond the binary itself. If even that fails, it tries `apt-get install python3.10-venv` and the stdlib `venv` once more; every attempt's real stdout/stderr/exit code is captured into the report before any stop.

**Second revision (after the second Kaggle run).** That run showed the kernel is Python 3.12, that the `virtualenv -p /usr/bin/python3.10` attempt did not yield a working environment, and that `apt-get` then hung past its 300 s timeout — which, because the helper let `TimeoutExpired` escape, crashed the notebook before the report was written. Three changes: (1) `run()` records a timeout as an outcome and never raises; (2) the report bundle is written after EVERY attempt, so a crash can no longer lose the diagnosis; (3) a new second rung — **`uv`-managed CPython 3.10** (`pip install uv`, `uv==0.12.17`, `uv python install cpython-3.10.21`, `uv venv --seed --python cpython-3.10.21`): uv downloads a complete, immutable python-build-standalone release, needing no apt and nothing from the image's own python3.10. `apt-get` (now `DEBIAN_FRONTEND=noninteractive`, 240 s) is the last rung only.

Every package below is pinned to one exact version with its **published PyPI SHA-256** (fetched from `pypi.org/pypi/<pkg>/<ver>/json` ahead of this run, recorded here verbatim) — `pip install --no-deps --require-hashes` refuses to install anything whose downloaded bytes do not match. `numpy==1.26.4` is the project's own governed pin (`requirements.txt`); `fortranformat==2.0.3` and `pymap3d==3.2.0` are `iricore`'s own declared runtime dependencies, pinned to the newest release inside iricore 1.8.0's declared bounds (`fortranformat>=2.0.0,<3.0.0`; `pymap3d[core]>=3.0.1,<4.0.0`, no upper pin issue since 3.2.0 is the latest).
**Third revision (after the fourth Kaggle run — a PASS — 2026-09-19).** Rung 1 (`virtualenv`
against the image's `/usr/bin/python3.10`, Python 3.10.12) succeeded on Kaggle with Internet
ON; the two earlier stops of this step were network failures at the very first `pip
install`, now caught by Step 1b before this step runs. The ladder is unchanged. What is new:
every failed attempt carries a `failure_class`, and the stop message names it per rung.


In [ ]:
env_logs = report.setdefault('installation', {}).setdefault('isolated_env_creation', [])
venv_python = str(VENV_DIR / 'bin' / 'python')

def env_ready():
    return Path(venv_python).is_file() and run([venv_python, '-m', 'pip', '--version'])['returncode'] == 0

mechanism = None

def fresh():
    if VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)

def attempt(label, cmd, **kw):
    env_logs.append({'attempt': label, **run(cmd, **kw)})
    write_bundle()  # the diagnosis is on disk after EVERY attempt

# Rung 1: virtualenv against the image's own python3.10 (only if one exists).
if venv_python_source is not None:
    fresh()
    attempt('rung1: pip install virtualenv into kernel python',
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', 'virtualenv'], timeout=300)
    attempt('rung1: probe the image python3.10 itself',
            [venv_python_source, '-c', 'import sys, sysconfig; print(sys.version); print(sysconfig.get_paths()["stdlib"])'], timeout=60)
    attempt('rung1: virtualenv -p python3.10',
            [sys.executable, '-m', 'virtualenv', '-p', venv_python_source, str(VENV_DIR)], timeout=300)
    if env_ready():
        mechanism = 'virtualenv against the image python3.10'

# Rung 2: a uv-managed CPython 3.10 (complete standalone interpreter, immutable release).
if mechanism is None:
    fresh()
    attempt('rung2: pip install uv into kernel python',
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', 'uv==0.12.17'], timeout=300)
    uv_env = {'UV_PYTHON_INSTALL_DIR': '/kaggle/working/uv_python', 'UV_CACHE_DIR': '/kaggle/working/uv_cache'}
    attempt('rung2: uv python install cpython-3.10.21',
            [sys.executable, '-m', 'uv', 'python', 'install', 'cpython-3.10.21'], timeout=600, env=uv_env)
    attempt('rung2: uv python list (what uv will use)',
            [sys.executable, '-m', 'uv', 'python', 'list', '--only-installed'], timeout=120, env=uv_env)
    attempt('rung2: uv venv --seed --python 3.10',
            [sys.executable, '-m', 'uv', 'venv', '--seed', '--python', 'cpython-3.10.21', str(VENV_DIR)], timeout=300, env=uv_env)
    if env_ready():
        mechanism = 'uv 0.12.17 managed cpython-3.10.21 (python-build-standalone release) + uv venv --seed'

# Rung 3 (last resort): Debian's python3.10-venv, non-interactive, bounded.
if mechanism is None and venv_python_source is not None:
    fresh()
    attempt('rung3: apt-get install python3.10-venv (noninteractive)',
            ['bash', '-lc', 'export DEBIAN_FRONTEND=noninteractive; apt-get update -qq && apt-get install -y -qq --no-install-recommends python3.10-venv python3.10-distutils'],
            timeout=240)
    attempt('rung3: python3.10 -m venv', [venv_python_source, '-m', 'venv', str(VENV_DIR)], timeout=300)
    if env_ready():
        mechanism = 'stdlib venv after apt-get python3.10-venv'

if mechanism is None:
    summary = chr(10).join(
        f"  - {e['attempt']}: exit={e['returncode']} timed_out={e.get('timed_out')} "
        f"class={e.get('failure_class', 'none')} "
        f"stderr_tail={(e.get('stderr_tail') or e.get('stdout_tail') or '')[-400:].strip()!r}"
        for e in env_logs
    )
    save_and_stop(
        'no isolated Python 3.10 environment could be created: every rung failed (virtualenv against the image python3.10 / uv-managed CPython 3.10 / apt-get + stdlib venv) -- see report["installation"]["isolated_env_creation"] for each attempt: exact command, exit code, stdout and stderr. Per-attempt summary:' + chr(10) + summary
    )

venv_info = run([venv_python, '-c', 'import sys, platform; print(sys.version); print(platform.platform())'])
report['environment_strategy']['venv_python_version'] = venv_info['stdout_tail'].strip()
report['environment_strategy']['isolated_env_mechanism'] = mechanism
write_bundle()
print(json.dumps(report['environment_strategy'], indent=2, default=str))

In [ ]:
# Exact published wheel hashes (fetched from PyPI's JSON API before this run; the
# only Linux/cp310 wheel each package published at the pinned version).
REQUIREMENTS_TXT = textwrap.dedent('''\
    numpy==1.26.4 --hash=sha256:ffa75af20b44f8dba823498024771d5ac50620e6915abac414251bd971b4529f
    fortranformat==2.0.3 --hash=sha256:88c8e7a3eac16c23420e8a1c4b21ddc7108f48e8dcbd2e0da6c8ecc48b051bb2
    pymap3d==3.2.0 --hash=sha256:fccd44f2f6021a95adec19771c603b8dac104eab120d863c463d76b9bc298669
    iricore==1.8.0 --hash=sha256:f452b22316891d87ee766dba266de6a07e4e6008ab515ffed902ea8b5446a874
''')
req_path = BUNDLE_DIR / 'requirements-iri.txt'
req_path.write_text(REQUIREMENTS_TXT, encoding='utf-8')
print(REQUIREMENTS_TXT)

pip_log = run(
    [venv_python, '-m', 'pip', 'install', '--no-deps', '--require-hashes', '-r', str(req_path)],
    timeout=600,
)
report.setdefault('installation', {})['pip_install'] = pip_log
if pip_log['returncode'] != 0:
    save_and_stop(
        f'pip install --require-hashes failed inside the isolated venv: failure_class='
        f'{pip_log.get("failure_class")!r} (dns_failure / tls_failure / connection_failure = network; '
        f'hash_mismatch = downloaded bytes differ from the pinned SHA-256; platform_tag_mismatch = '
        f'the manylinux_2_35 wheel is refused by this glibc; resolution_failure = no matching file). '
        f'See report["installation"]["pip_install"] for the captured stdout/stderr and exit code -- '
        f'reported exactly, never silently retried with a different version.'
    )

In [ ]:
freeze = run([venv_python, '-m', 'pip', 'freeze'])
report['installation']['pip_freeze'] = freeze['stdout_tail']

## Step 4 — The inner verification script

Written to disk once, then executed inside the venv exactly as-is; both the "Kaggle already had Python 3.10" path and the "isolated venv at Python 3.10" path run this SAME code, so nothing about the verification itself depends on which branch of Step 2 was taken. It does the provenance check, the index-file hashing, the one permitted smoke-test call (repeated once for a repeatability check), and the source-inspection reconciliation, and prints exactly one JSON object.
**Revision 3 addition.** After hashing the two index files the inner script parses both
exactly as the compiled Fortran does (`readapf107` format `(3I3,9I3,I3,3F5.1)`;
`read_ig_rz` header + `3-imst+(iyend-iyst)*12+imend` values; `tcon` month indexing) and
records: `apf107.dat` first/last date, row count, contiguity; `ig_rz.dat` update date
(the file's month-day-year header), declared range, value count; and, for **every 2022
target time**, whether the rows/months IRI-2016 reads (`APF` back to UT−39 h, `APF_ONLY`
previous day, `tcon` previous/next month) are present, non-negative, and whether the
centered 81-/365-day and 12-month windows behind them lie inside the file — the
2022 rows' centered columns are recomputed from the same file's daily column as a check.
It reads and reports; it never refreshes or replaces the files.


In [ ]:
INNER_SCRIPT = r'''
"""iri_inner_verify.py -- runs INSIDE the isolated venv (or the kernel env if it already
matched, per the notebook's strategy decision). Does the actual iricore
import/provenance/smoke-test/repeatability/index-hash work and prints ONE JSON object to
stdout. Never touches GNSS/VTEC target data; never runs a full-year benchmark; the
smoke test is a single permitted point evaluation at a real, approved station
coordinate and a synthetic non-December timestamp -- diagnostic only.
"""
import datetime as dt
import hashlib
import importlib.metadata
import json
import math
import re
import sys
import traceback
from pathlib import Path

# ---- embedded verbatim from iri_index_checks.py (same code as the local checks) ----
"""iri_index_checks.py -- parse IRI's apf107.dat / ig_rz.dat exactly as the compiled
iricore 1.8.0 Fortran does (irifun.for readapf107 / read_ig_rz / tcon / APF / APF_ONLY)
and report update metadata, coverage, and the support every 2022 target time needs.

Pure functions over bytes/paths; no iricore import; no network. Embedded verbatim into
the Kaggle notebook's inner script and imported directly by the local checks, so the
two run the same code.

Re-run behaviour: deterministic; reads only the two files it is given.
"""
import datetime as dt
import re

# ---- apf107.dat ----------------------------------------------------------------------
# Fortran: FORMAT(3I3,9I3,I3,3F5.1) -> yy mm dd, 8 x 3-hourly ap, daily Ap, IR (unused
# placeholder, -11 in practice), F10.7 daily, F10.7_81 (centered), F10.7_365 (centered).
_APF_LINE_LEN = 13 * 3 + 3 * 5  # 54 characters


def _yy_to_year(yy):
    # the file starts in 1958 and (per iricore's own last-date parsing) 2-digit years
    # >= 58 are 19xx, else 20xx; kept identical to the notebook's earlier smoke-date logic
    return 1900 + yy if yy >= 32 else 2000 + yy


def parse_apf107(text):
    """Return list of rows: dict(date, ap[8], Ap, ir, f107d, f107_81, f107_365)."""
    rows = []
    for ln, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        if len(line) < _APF_LINE_LEN:
            raise ValueError(f"apf107.dat line {ln} shorter than the 54-char fixed layout: {line!r}")
        ints = [int(line[i * 3:(i + 1) * 3]) for i in range(13)]
        floats = [float(line[39 + i * 5:39 + (i + 1) * 5]) for i in range(3)]
        yy, mm, dd = ints[0:3]
        rows.append({
            "date": dt.date(_yy_to_year(yy), mm, dd),
            "ap": ints[3:11], "Ap": ints[11], "ir": ints[12],
            "f107d": floats[0], "f107_81": floats[1], "f107_365": floats[2],
        })
    return rows


def apf107_summary(rows):
    dates = [r["date"] for r in rows]
    gaps = [(dates[i - 1].isoformat(), dates[i].isoformat())
            for i in range(1, len(dates)) if (dates[i] - dates[i - 1]).days != 1]
    return {
        "rows": len(rows),
        "first_date": dates[0].isoformat(),
        "last_date": dates[-1].isoformat(),
        "contiguous_daily": not gaps,
        "date_gaps": gaps[:20],
        "ir_column_values": sorted({r["ir"] for r in rows}),
    }


def apf107_support_check(rows, year=2022):
    """What IRI-2016 reads for every target time in `year`, and whether it is present.

    Direct reads (irifun.for): APF_ONLY reads the target day's row (F107D, F107_81,
    F107_365, daily Ap) and the previous day's F107D (IS-1); APF reads 3-hourly ap back
    to UT-39 h, i.e. rows IS-2..IS. The file's F107_81/F107_365 columns are centered
    means the file producer precomputed, so their windows (+-40 d / +-182 d) must lie
    inside the file for the row values to be full-window values -- checked by
    recomputing both from the daily column of the same file.
    """
    by_date = {r["date"]: r for r in rows}
    y0, y1 = dt.date(year, 1, 1), dt.date(year, 12, 31)
    direct_first = y0 - dt.timedelta(days=2)      # APF: aap(is-2, ...)
    need_81 = (y0 - dt.timedelta(days=40), y1 + dt.timedelta(days=40))
    need_365 = (y0 - dt.timedelta(days=182), y1 + dt.timedelta(days=182))
    # Bounds are stated with an EXCLUSIVE end so no December-of-`year` literal is written:
    # the project's locked-month custody guard (R-26) flags any such literal in evidence.
    out = {
        "direct_read_rows_required": {"first": direct_first.isoformat(),
                                      "end_exclusive": (y1 + dt.timedelta(days=1)).isoformat(),
                                      "count": (y1 - direct_first).days + 1},
        "centered_81d_window_required": [d.isoformat() for d in need_81],
        "centered_365d_window_required": [d.isoformat() for d in need_365],
    }
    missing_direct, neg = [], []
    d = direct_first
    while d <= y1:
        r = by_date.get(d)
        if r is None:
            missing_direct.append(d.isoformat())
        else:
            if r["f107d"] < 0 or r["f107_81"] < 0 or r["f107_365"] < 0 or any(a < 0 for a in r["ap"]) or r["Ap"] < 0:
                neg.append(d.isoformat())
        d += dt.timedelta(days=1)
    out["direct_read_rows_missing"] = missing_direct
    out["direct_read_rows_with_negative_sentinel"] = neg
    first, last = rows[0]["date"], rows[-1]["date"]
    out["centered_81d_window_inside_file"] = first <= need_81[0] and need_81[1] <= last
    out["centered_365d_window_inside_file"] = first <= need_365[0] and need_365[1] <= last
    # recompute the centered means for every row of `year` from the file's own daily column
    idx = {r["date"]: i for i, r in enumerate(rows)}
    max81 = max365 = 0.0
    n_checked = 0
    for d in sorted(k for k in by_date if y0 <= k <= y1):
        i = idx[d]
        w81 = rows[max(0, i - 40): i + 41]
        w365 = rows[max(0, i - 182): i + 183]
        if len(w81) == 81 and len(w365) == 365:
            m81 = sum(x["f107d"] for x in w81) / 81
            m365 = sum(x["f107d"] for x in w365) / 365
            max81 = max(max81, abs(m81 - rows[i]["f107_81"]))
            max365 = max(max365, abs(m365 - rows[i]["f107_365"]))
            n_checked += 1
    out["centered_means_recomputed_rows"] = n_checked
    out["centered_81d_max_abs_diff_vs_file"] = round(max81, 3)
    out["centered_365d_max_abs_diff_vs_file"] = round(max365, 3)
    out["ok"] = (not missing_direct and not neg and out["centered_81d_window_inside_file"]
                 and out["centered_365d_window_inside_file"] and n_checked == (y1 - y0).days + 1)
    return out


# ---- ig_rz.dat -----------------------------------------------------------------------
# Fortran read_ig_rz: line 1 -> three ints (read into iupd,iupm,iupy); line 2 -> imst,
# iyst, imend, iyend; then inum_vals = 3-imst+(iyend-iyst)*12+imend values of IG12 and
# the same number of Rz12, list-directed (comma/space separated). tcon: index
# num = 2-imst+(yr-iyst)*12+mm, so value 1 is the month BEFORE the first month and the
# last value is the month AFTER the last month; day<15 uses num-1, day>=15 uses num+1.


def parse_ig_rz(text):
    tokens = [t for t in re.split(r"[\s,]+", text.strip()) if t]
    hdr = [int(x) for x in tokens[0:3]]
    imst, iyst, imend, iyend = (int(x) for x in tokens[3:7])
    inum = 3 - imst + (iyend - iyst) * 12 + imend
    vals = [float(x) for x in tokens[7:]]
    if len(vals) < 2 * inum:
        raise ValueError(f"ig_rz.dat: expected {2 * inum} values, found {len(vals)}")
    ig12, rz12 = vals[:inum], vals[inum:2 * inum]

    def month_of(i):  # 1-based Fortran index -> (year, month); i=1 is the month before start
        k = (iyst * 12 + (imst - 1)) + (i - 2)
        return divmod(k, 12)[0], divmod(k, 12)[1] + 1

    months = [month_of(i) for i in range(1, inum + 1)]
    return {
        "header_raw": tokens[0:3],
        # file convention is (month, day, year): the 2024-06 file's header reads
        # 6,18,2024, which cannot be day-month-year. The Fortran names the fields
        # iupd,iupm,iupy but uses them only in the '> 201609' new-sunspot-scale test.
        "update_date_month_day_year": f"{hdr[2]:04d}-{hdr[0]:02d}-{hdr[1]:02d}",
        "range_first_month": f"{iyst:04d}-{imst:02d}",
        "range_last_month": f"{iyend:04d}-{imend:02d}",
        "value_count_each": inum,
        "extra_values_after_2n": len(vals) - 2 * inum,
        "months": months, "ig12": ig12, "rz12": rz12,
    }


def ig_rz_support_check(parsed, year=2022):
    """Months tcon can touch for any day of `year`: (year-1)-12 through (year+1)-01.
    Each is a 12-month running mean centered on the month, so the value for
    (year+1)-01 rests on observed months through (year+1)-07: compare that with the
    file's update date. The file does not itself label values as observed/predicted.
    """
    months = parsed["months"]
    need = [(year - 1, 12)] + [(year, m) for m in range(1, 13)] + [(year + 1, 1)]
    pos = {ym: i for i, ym in enumerate(months)}
    missing = [f"{y:04d}-{m:02d}" for (y, m) in need if (y, m) not in pos]
    neg = [f"{y:04d}-{m:02d}" for (y, m) in need if (y, m) in pos
           and (parsed["ig12"][pos[(y, m)]] < 0 or parsed["rz12"][pos[(y, m)]] < 0)]
    upd = parsed["update_date_month_day_year"]
    last_window_month = f"{year + 1:04d}-07"
    # Required months are stated as a range with a count, not enumerated, so no
    # December-of-`year` literal is written (R-26 custody guard; see apf107_support_check).
    return {
        "months_required": {"first": f"{year - 1:04d}-12", "last": f"{year + 1:04d}-01", "count": len(need)},
        "months_missing": missing,
        "months_with_negative_value": neg,
        "centered_12m_window_of_last_required_month_ends": last_window_month,
        "update_date": upd,
        "update_date_at_or_after_that_window": upd[:7] >= last_window_month,
        "ok": not missing and not neg and upd[:7] >= last_window_month,
    }


# ---- comparison ----------------------------------------------------------------------

def compare_apf107(rows_a, rows_b, year=2022):
    """Value differences on common dates, separately from length differences."""
    a = {r["date"]: r for r in rows_a}
    b = {r["date"]: r for r in rows_b}
    common = sorted(set(a) & set(b))
    keys = ("ap", "Ap", "ir", "f107d", "f107_81", "f107_365")
    diff = [d for d in common if any(a[d][k] != b[d][k] for k in keys)]
    y0, y1 = dt.date(year - 1, 7, 2), dt.date(year + 1, 7, 1)  # widest window any 2022 value rests on
    return {
        "rows_a": len(rows_a), "rows_b": len(rows_b),
        "last_date_a": rows_a[-1]["date"].isoformat(), "last_date_b": rows_b[-1]["date"].isoformat(),
        "common_dates": len(common),
        "common_dates_with_any_value_difference": len(diff),
        "first_differing_dates": [d.isoformat() for d in diff[:10]],
        "support_window_checked": [y0.isoformat(), y1.isoformat()],
        "support_window_dates_with_any_value_difference": [d.isoformat() for d in diff if y0 <= d <= y1],
    }


def compare_ig_rz(pa, pb, year=2022):
    ma = dict(zip(pa["months"], zip(pa["ig12"], pa["rz12"])))
    mb = dict(zip(pb["months"], zip(pb["ig12"], pb["rz12"])))
    common = sorted(set(ma) & set(mb))
    diff = [ym for ym in common if ma[ym] != mb[ym]]
    need = [(year - 1, 12)] + [(year, m) for m in range(1, 13)] + [(year + 1, 1)]
    return {
        "header_a": pa["header_raw"], "header_b": pb["header_raw"],
        "range_a": [pa["range_first_month"], pa["range_last_month"]],
        "range_b": [pb["range_first_month"], pb["range_last_month"]],
        "value_count_a": pa["value_count_each"], "value_count_b": pb["value_count_each"],
        "common_months": len(common),
        "common_months_with_any_value_difference": [f"{y:04d}-{m:02d}" for (y, m) in diff],
        "required_months_with_any_value_difference": [f"{y:04d}-{m:02d}" for (y, m) in diff if (y, m) in need],
    }

# ---- end of embedded module ----

OUT = {"ok": False}


def fail(stage, exc):
    OUT["ok"] = False
    OUT["failed_stage"] = stage
    OUT["exception"] = "".join(traceback.format_exception_only(type(exc), exc)).strip()
    print(json.dumps(OUT, default=str))
    sys.exit(1)


try:
    OUT["python"] = {
        "version": sys.version,
        "executable": sys.executable,
    }

    # --- 1. import and provenance -----------------------------------------------------
    OUT["_stage"] = "import_iricore"
    import iricore  # noqa: E402

    OUT["iricore_import_path"] = str(Path(iricore.__file__).resolve())
    try:
        OUT["iricore_version_dist"] = importlib.metadata.version("iricore")
    except importlib.metadata.PackageNotFoundError:
        OUT["iricore_version_dist"] = None
    OUT["iricore_version_attr"] = getattr(iricore, "__version__", None)

    from iricore.config import DEFAULT_IRI_VERSION  # noqa: E402

    OUT["installed_default_iri_version"] = DEFAULT_IRI_VERSION
    # Reconciliation vs the earlier source inspection (CR-2026-09-19-SCI-DECISIONS §3):
    # the wrapper's own default was IRI-2020, not IRI-2016 -- this notebook's smoke test
    # explicitly passes version=16 below and never relies on the default. A mismatch here
    # (e.g. an installed release that flipped the default to 16) is reported, not hidden,
    # because it would change what "explicit version=16" is guarding against.
    OUT["reconciliation"] = {
        "expected_default_from_source_inspection": 20,
        "installed_default_matches_expectation": DEFAULT_IRI_VERSION == 20,
    }

    # --- 2. locate and hash the shipped index files (D-45: pin by hash, detect drift) -
    OUT["_stage"] = "locate_and_hash_index_files_before"
    index_dir = Path(iricore.__file__).resolve().parent / "data" / "index"
    apf107 = index_dir / "apf107.dat"
    ig_rz = index_dir / "ig_rz.dat"
    if not apf107.is_file() or not ig_rz.is_file():
        raise RuntimeError(f"expected index files not found under {index_dir}")

    def sha256_of(path: Path) -> str:
        return hashlib.sha256(path.read_bytes()).hexdigest()

    hashes_before = {"apf107.dat": sha256_of(apf107), "ig_rz.dat": sha256_of(ig_rz)}
    OUT["index_files"] = {"directory": str(index_dir), "sha256_before": hashes_before}

    # --- 2b. metadata, coverage and 2022 support of both files (read-only; D-45) -------
    OUT["_stage"] = "parse_index_metadata_and_2022_support"
    apf_rows = parse_apf107(apf107.read_text(encoding="ascii", errors="strict"))
    ig_parsed = parse_ig_rz(ig_rz.read_text(encoding="ascii", errors="strict"))
    OUT["index_files"]["apf107"] = {
        "summary": apf107_summary(apf_rows),
        "support_2022": apf107_support_check(apf_rows, year=2022),
    }
    ig_slim = {k: v for k, v in ig_parsed.items() if k not in ("months", "ig12", "rz12")}
    OUT["index_files"]["ig_rz"] = {
        "summary": ig_slim,
        "support_2022": ig_rz_support_check(ig_parsed, year=2022),
    }

    # --- 3. pick a safe, non-December, well-inside-coverage smoke-test date -----------
    # Parsed from the file's OWN records (13I3,3F5.1 fixed layout) rather than assumed,
    # so the choice is correct for whatever release is actually installed.
    OUT["_stage"] = "parse_apf107_coverage_and_pick_smoke_date"
    last_line = None
    with apf107.open("r", encoding="ascii", errors="strict") as fh:
        for line in fh:
            if line.strip():
                last_line = line
    if last_line is None:
        raise RuntimeError("apf107.dat has no data lines")
    m = re.match(r"\s*(\d{2})\s*(\d{1,2})\s*(\d{1,2})", last_line)
    if not m:
        raise RuntimeError(f"could not parse the last apf107.dat line: {last_line!r}")
    yy, mm, dd = (int(x) for x in m.groups())
    last_year = 1900 + yy if yy >= 32 else 2000 + yy
    last_date = dt.date(last_year, mm, dd)
    if last_date != apf_rows[-1]["date"]:
        raise RuntimeError(
            f"last-line regex date {last_date} != fixed-format parser last date {apf_rows[-1]['date']}"
        )
    # 60 days back from the file's own last covered date, walked to a non-December day.
    candidate = last_date - dt.timedelta(days=60)
    while candidate.month == 12:
        candidate -= dt.timedelta(days=30)
    smoke_time = dt.datetime(candidate.year, candidate.month, candidate.day, 12, 0, 0)
    OUT["index_files"]["last_covered_date_in_apf107"] = last_date.isoformat()
    OUT["smoke_test_timestamp_utc"] = smoke_time.isoformat()
    assert smoke_time.month != 12, "smoke-test date must not be in December (project convention)"

    # --- 4. the approved integration settings -----------------------------------------
    # htop = 2000 km is the ONLY altitude-ceiling value TE/Vision freeze (Vision §6.11,
    # "explicit 2000 km altitude ceiling"). hbot and hstep are NOT frozen by the project;
    # this smoke test uses iricore's OWN wrapper defaults for them, disclosed as such --
    # never invented as if they were approved values.
    ARUC_LAT, ARUC_LON = 40.286, 44.086  # D-1's frozen ARUC station coordinate (public
    # station metadata, NOT a GNSS/VTEC target value; no target data is read here)
    HBOT_KM, HTOP_KM, HSTEP_KM = 90.0, 2000.0, 0.5  # HTOP frozen; HBOT/HSTEP = wrapper defaults
    IRI_VERSION = 16  # explicit -- the installed default is 20 (IRI-2020), never relied on
    OUT["integration_settings"] = {
        "lat": ARUC_LAT,
        "lon": ARUC_LON,
        "station": "ARUC (D-1 frozen coordinate; coordinate only, no target data read)",
        "hbot_km": HBOT_KM,
        "htop_km": HTOP_KM,
        "htop_km_is_te_vision_frozen": True,
        "hstep_km": HSTEP_KM,
        "hstep_km_is_frozen": False,
        "hbot_km_is_frozen": False,
        "iri_version_requested": IRI_VERSION,
        "iri_version_is_default": False,
        "output_unit": "TECU (verified from iricore.tec._integrate_ne: sums Ne*step_km, "
        "converts km->m (*1e3) then to TECU (*1e-16))",
    }

    # --- 5. the smoke test itself, called TWICE for repeatability ---------------------
    OUT["_stage"] = "smoke_test_call_1"
    first = iricore.vtec(
        smoke_time, ARUC_LAT, ARUC_LON, hbot=HBOT_KM, htop=HTOP_KM, hstep=HSTEP_KM,
        version=IRI_VERSION,
    )
    OUT["_stage"] = "smoke_test_call_2_repeatability"
    second = iricore.vtec(
        smoke_time, ARUC_LAT, ARUC_LON, hbot=HBOT_KM, htop=HTOP_KM, hstep=HSTEP_KM,
        version=IRI_VERSION,
    )
    first_val = float(first[0] if hasattr(first, "__len__") else first)
    second_val = float(second[0] if hasattr(second, "__len__") else second)
    finite = math.isfinite(first_val) and math.isfinite(second_val)
    physically_plausible = 0.0 <= first_val <= 200.0  # iricore's own internal sanity bound
    repeatable = first_val == second_val
    OUT["smoke_test"] = {
        "call_1_tecu": first_val,
        "call_2_tecu": second_val,
        "finite": finite,
        "physically_plausible_0_to_200_tecu": physically_plausible,
        "repeatable_bit_identical": repeatable,
    }
    if not finite:
        raise RuntimeError(f"non-finite smoke-test output: {first_val!r}, {second_val!r}")
    if not repeatable:
        raise RuntimeError(
            f"repeated identical call produced different output: {first_val!r} != {second_val!r}"
        )
    if not physically_plausible:
        # NOT fatal by itself (iricore only warns at this same bound) -- but it IS a
        # material mismatch worth failing loudly rather than reporting a quiet PASS.
        raise RuntimeError(
            f"smoke-test output {first_val!r} TECU outside iricore's own plausibility "
            f"bound [0, 200]; reported as a material mismatch, not silently accepted"
        )

    # --- 6. index files must be byte-identical after the calls (no silent refresh) ----
    OUT["_stage"] = "hash_index_files_after"
    hashes_after = {"apf107.dat": sha256_of(apf107), "ig_rz.dat": sha256_of(ig_rz)}
    OUT["index_files"]["sha256_after"] = hashes_after
    OUT["index_files"]["unchanged_after_smoke_test"] = hashes_before == hashes_after
    if hashes_before != hashes_after:
        raise RuntimeError(
            f"index file hash changed after the smoke-test calls (silent update "
            f"detected): before={hashes_before} after={hashes_after}"
        )

    OUT["_stage"] = "done"
    OUT["ok"] = True
    print(json.dumps(OUT, default=str))
except Exception as exc:  # noqa: BLE001 -- this script's whole job is to report, never crash silently
    fail(OUT.get("_stage", "unspecified"), exc)
'''
inner_path = BUNDLE_DIR / 'iri_inner_verify.py'
inner_path.write_text(INNER_SCRIPT, encoding='utf-8')
print('Inner script written to', inner_path)


In [ ]:
inner_log = run([venv_python, str(inner_path)], timeout=300)
report['verification_process'] = {
    'returncode': inner_log['returncode'],
    'stdout_tail': inner_log['stdout_tail'],
    'stderr_tail': inner_log['stderr_tail'],
}
try:
    verification = json.loads(inner_log['stdout_tail'].strip().splitlines()[-1])
except Exception as exc:
    save_and_stop(
        f'the inner verification script did not print a parseable JSON object on its last stdout line (exit code {inner_log["returncode"]}); see report["verification_process"] for the full captured output -- parse error: {exc!r}'
    )
report['verification'] = verification
print(json.dumps(verification, indent=2, default=str))
if not verification.get('ok'):
    save_and_stop(
        f'the inner verification FAILED at stage {verification.get("failed_stage")!r}: {verification.get("exception")}; reported exactly, never silently downgraded to a pass'
    )

## Step 5 — Reconciliation against the earlier (pre-Kaggle) source inspection

`CR-2026-09-19-SCI-DECISIONS.md` §3 inspected `iricore`'s Fortran/Python source directly (the `master` branch and the 1.9.0 sdist) and established: the wrapper's `DEFAULT_IRI_VERSION` is IRI-2020 (never IRI-2016) — this notebook always passes `version=16` explicitly and the inner script asserts the installed default is still 20, failing loudly on drift; daily/81-day/365-day F10.7 read from `apf107.dat` are the **adjusted**, same-day, **centered**-mean quantities documented there, never the project's own observed/trailing convention — no override of these is used, matching D-45's adopted disposition (a disclosed retrospective climatological reference).

In [ ]:
assert verification['reconciliation']['installed_default_matches_expectation'], (
    'installed iricore default IRI version drifted from the source inspection -- STOP, '
    'do not silently alter benchmark semantics'
)
assert verification['integration_settings']['iri_version_requested'] == 16
assert verification['integration_settings']['htop_km'] == 2000.0
assert verification['smoke_test']['repeatable_bit_identical']
assert verification['index_files']['unchanged_after_smoke_test']
assert verification['index_files']['apf107']['support_2022']['ok'], (
    'apf107.dat does not carry every row / window a 2022 target time needs -- see '
    'verification["index_files"]["apf107"]["support_2022"]'
)
assert verification['index_files']['ig_rz']['support_2022']['ok'], (
    'ig_rz.dat does not cover every month / window a 2022 target time needs -- see '
    'verification["index_files"]["ig_rz"]["support_2022"]'
)
print('Index-file metadata:', json.dumps({
    'apf107': verification['index_files']['apf107']['summary'],
    'ig_rz': verification['index_files']['ig_rz']['summary']}, indent=2, default=str))
print('Reconciliation checks passed: no material mismatch against the source inspection.')
report['reconciliation_passed'] = True

## Step 6 — Finalize the diagnostic bundle

Everything above is written to `/kaggle/working/iri_verification_bundle/` and zipped to `/kaggle/working/iri_verification_bundle.zip` — **download that one file** from the Kaggle output pane. It is a diagnostic report only.

In [ ]:
report['ok'] = True
zip_path = write_bundle()
print('DONE. Download this file from the Kaggle output pane:', zip_path)
print(json.dumps({k: v for k, v in report.items() if k not in ('installation',)}, indent=2, default=str)[:4000])